# XGBoost : exploration

Ce notebook teste XGBoost avec les mêmes règles que les autres modèles : découpage commun, validation croisée stratifiée à 5 plis sur le jeu d'entraînement, PR-AUC comme métrique principale et seuil de décision à 0,5 (on optimisera celui du modèle qu'on retiendra à la fin). Le jeu de test n'est pas utilisé, il reste de côté pour l'évaluation finale.

XGBoost fait du gradient boosting : des petits arbres entraînés les uns après les autres, chacun corrige les erreurs des précédents. On part des réglages par défaut, puis on règle la profondeur des arbres, le taux d'apprentissage et le nombre d'arbres. Chaque essai est enregistré dans MLflow.

## Importations et pré-réglages

In [ ]:
import mlflow
import pandas as pd
from mlflow.models import infer_signature
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

from base_commune.outils import (
    RANDOM_STATE,
    charger_donnees,
    configurer_mlflow,
    decouper_donnees,
    evaluer_modele,
    exporter_runs,
    tags_reproductibilite,
)

AUTEUR = "david"
CREER_LES_RUNS = False  # True pour relancer les runs

In [ ]:
df_brut = charger_donnees()
X_train, X_test, y_train, y_test = decouper_donnees(df_brut)
print(X_train.shape, X_test.shape)

## Pipeline

Les arbres n'ont pas besoin de standardisation, on encode juste Geography et Gender.

In [ ]:
vars_num = ["CreditScore", "Age", "Tenure", "Balance", "NumOfProducts", "EstimatedSalary",
            "HasCrCard", "IsActiveMember"]
vars_cate = ["Geography", "Gender"]


def construire_xgb(n_estimators=100, max_depth=6, learning_rate=0.3, scale_pos_weight=1):
    preprocesseur = ColumnTransformer([
        ("cate", OneHotEncoder(handle_unknown="ignore"), vars_cate),
        ("num", "passthrough", vars_num),
    ], remainder="drop")
    return Pipeline([
        ("preprocesseur", preprocesseur),
        ("xgb", XGBClassifier(n_estimators=n_estimators, max_depth=max_depth,
                              learning_rate=learning_rate, scale_pos_weight=scale_pos_weight,
                              random_state=RANDOM_STATE, n_jobs=-1)),
    ])

## Suivi MLflow

In [ ]:
configurer_mlflow("churn_bancaire")


def enregistrer_run(modele, nom_run, params, logger_modele=False, stage="exploration"):
    with mlflow.start_run(run_name=nom_run):
        mlflow.set_tags({"model_family": "xgboost", "author": AUTEUR, "stage": stage,
                         **tags_reproductibilite()})
        mlflow.log_params({**params, "features": ",".join(vars_cate + vars_num),
                           "cv_folds": 5, "random_state": RANDOM_STATE,
                           "test_size": 0.2, "n_train": len(X_train)})
        metriques = evaluer_modele(modele, X_train, y_train)
        mlflow.log_metrics(metriques)
        if logger_modele:
            modele.fit(X_train, y_train)
            signature = infer_signature(X_train, modele.predict_proba(X_train)[:, 1])
            mlflow.sklearn.log_model(modele, name="model", signature=signature,
                                     input_example=X_train.head(3))
    return metriques


def afficher(nom, m):
    print(f"{nom:35s} PR-AUC {m['cv_pr_auc_mean']:.3f} (± {m['cv_pr_auc_std']:.3f})"
          f"  train {m['train_pr_auc_mean']:.3f}  recall {m['cv_recall_mean']:.3f}")

## Réglages par défaut

100 arbres de profondeur 6 avec un taux d'apprentissage de 0,3.

In [ ]:
if CREER_LES_RUNS:
    params = {"n_estimators": 100, "max_depth": 6, "learning_rate": 0.3, "scale_pos_weight": 1}
    m = enregistrer_run(construire_xgb(**params), "xgb_defaut", params)
    afficher("xgb_defaut", m)

PR-AUC de 0,665 en validation mais 0,987 sur le train : avec des arbres de profondeur 6, le modèle apprend le train presque par cœur.

## Réglage des arbres

On teste des arbres moins profonds, un taux d'apprentissage plus petit et plus ou moins d'arbres.

In [ ]:
if CREER_LES_RUNS:
    for max_depth in [2, 3, 4]:
        for learning_rate in [0.05, 0.1]:
            for n_estimators in [100, 300]:
                params = {"n_estimators": n_estimators, "max_depth": max_depth,
                          "learning_rate": learning_rate, "scale_pos_weight": 1}
                nom = f"xgb_depth{max_depth}_lr{learning_rate}_n{n_estimators}"
                m = enregistrer_run(construire_xgb(**params), nom, params)
                afficher(nom, m)

Avec des arbres moins profonds, l'écart entre le train et la validation se réduit et le PR-AUC monte autour de 0,69-0,70. Beaucoup de configurations se valent, les écarts restent plus petits que l'écart-type entre les plis.

On garde 300 arbres de profondeur 3 avec un taux d'apprentissage de 0,05 : PR-AUC de 0,699 avec 0,760 sur le train. La profondeur 4 donne 0,700, mais avec plus d'écart avec le train (0,806).

## Déséquilibre des classes

`scale_pos_weight` donne plus de poids aux churners. On met le rapport non-churners / churners du train, soit environ 3,9.

In [ ]:
POIDS = round((y_train == 0).sum() / (y_train == 1).sum(), 2)
print(POIDS)

if CREER_LES_RUNS:
    params = {"n_estimators": 300, "max_depth": 3, "learning_rate": 0.05,
              "scale_pos_weight": POIDS}
    m = enregistrer_run(construire_xgb(**params), "xgb_depth3_lr0.05_n300_poids", params)
    afficher("xgb_depth3_lr0.05_n300_poids", m)

Le PR-AUC ne bouge pas (0,701) mais le recall passe de 46 % à 74 %. Comme pour class_weight sur la régression logistique, ça revient à baisser le seuil, donc on garde scale_pos_weight à 1 et on réglera le seuil à la fin.

## Export des runs

In [ ]:
chemin = exporter_runs("churn_bancaire", AUTEUR)
print(chemin)

## Importance des variables du modèle retenu

L'importance indique à quel point chaque variable sert dans les découpages des arbres.

In [ ]:
modele_retenu = construire_xgb(n_estimators=300, max_depth=3, learning_rate=0.05)
modele_retenu.fit(X_train, y_train)

noms = modele_retenu.named_steps["preprocesseur"].get_feature_names_out()
importances = pd.Series(modele_retenu.named_steps["xgb"].feature_importances_, index=noms)
importances.sort_values(ascending=False).round(3)

NumOfProducts, IsActiveMember et l'âge sont les trois variables les plus utilisées, puis l'Allemagne. On retrouve les mêmes variables que dans l'EDA et dans les coefficients de la régression logistique. Tenure, CreditScore, EstimatedSalary et HasCrCard servent peu.

## Conclusion

Avec des arbres peu profonds, XGBoost arrive à 0,699 de PR-AUC : c'est le meilleur score pour l'instant, devant le Random Forest (0,68), la régression logistique (0,665) et le KNN (0,621).